In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import gymnasium as gym
import torch

sys.path.insert(0, os.path.abspath(".."))

from config.match_config import MatchConfig, PlayerSlot, PlayerStats
from src.bots.heuristic_bot import TeamHeuristicCoordinator
from src.engine.controllers import HeuristicBotController
from src.engine.modes.classic_mode import ClassicMatchMode
from src.rl.env_wrapper import HaxballGymEnv
from src.rl.ppo_core import ActorCritic
from src.rl.reset_strategies import MatchKickoffReset
from src.rl.reward_shapers import Stage2ScratchReward
from src.rl.trainer import train_ppo_vectorized


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Device: {device}")

NUM_ENVS = 16
OBS_DIM = 80

⚡ Device: cuda


In [2]:
def make_fixed_sparring_env(
    opponent_accel: float,
    max_steps: int = 600,
):
    def _init():
        blue_coord = TeamHeuristicCoordinator(team="blue")
        roster = [
            PlayerSlot(
                team="red", stats=PlayerStats(name="RL_Agent", accel=3200.0)
            ),
            PlayerSlot(
                team="blue",
                stats=PlayerStats(name="Sparring_Bot", accel=opponent_accel),
                controller=HeuristicBotController(blue_coord),
            ),
        ]

        match_cfg = MatchConfig(
            mode=ClassicMatchMode(
                time_limit=10.0, score_limit=1, kickoff_timeout=5.0
            ),
            roster=roster,
            time_limit=10.0,
            score_limit=1,
        )

        return HaxballGymEnv(
            match_config=match_cfg,
            reward_shaper=Stage2ScratchReward(team="red"),
            reset_strategy=MatchKickoffReset(),  # Fixed positions
            max_steps=max_steps,
        )

    return _init


In [ ]:

# ─────────────────────────────────────────────────────────────
# PHASE A: Slow Bot (1600 Accel) - Training From Scratch
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("🌱 PHASE A: Training From Scratch vs Slow Bot (Accel: 1600.0)")
print("=" * 60)

train_envs_a = gym.vector.AsyncVectorEnv(
    [make_fixed_sparring_env(opponent_accel=1600.0) for _ in range(NUM_ENVS)]
)
eval_env_a = make_fixed_sparring_env(opponent_accel=1600.0)()

# Fresh Model Initialization
model = ActorCritic(obs_dim=OBS_DIM).to(device)

train_ppo_vectorized(
    envs=train_envs_a,
    eval_env=eval_env_a,
    model=model,
    device=device,
    total_timesteps=10_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    eval_freq=100_000,
    eval_episodes=50,
    save_dir="models/stage2_scratch/phaseA",
    lr_initial=3e-4,  
    lr_final=1e-5,
    ent_coef_initial=0.015,
    ent_coef_final=0.001,
)

train_envs_a.close()
eval_env_a.close()


# ─────────────────────────────────────────────────────────────
# PHASE B: Medium Bot (2300 Accel) - Intermediate Speed
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("🥊 PHASE B: Medium Bot (Accel: 2300.0)")
print("=" * 60)

train_envs_b = gym.vector.AsyncVectorEnv(
    [make_fixed_sparring_env(opponent_accel=2300.0) for _ in range(NUM_ENVS)]
)
eval_env_b = make_fixed_sparring_env(opponent_accel=2300.0)()

phase_a_ckpt = "models/stage2_scratch/phaseA/best_model.pt"
model.load_state_dict(
    torch.load(phase_a_ckpt, map_location=device, weights_only=False)
)
print(f"✅ Loaded Phase A Checkpoint from {phase_a_ckpt}")

train_ppo_vectorized(
    envs=train_envs_b,
    eval_env=eval_env_b,
    model=model,
    device=device,
    total_timesteps=10_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    eval_freq=100_000,
    eval_episodes=50,
    save_dir="models/stage2_scratch/phaseB",
    lr_initial=1e-4,
    lr_final=1e-5,
    ent_coef_initial=0.005,
    ent_coef_final=0.0005,
)

train_envs_b.close()
eval_env_b.close()


# ─────────────────────────────────────────────────────────────
# PHASE C: Full Match Intensity Bot (3000 Accel)
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("🔥 PHASE C: Full Speed Bot (Accel: 3000.0)")
print("=" * 60)

train_envs_c = gym.vector.AsyncVectorEnv(
    [make_fixed_sparring_env(opponent_accel=3000.0) for _ in range(NUM_ENVS)]
)
eval_env_c = make_fixed_sparring_env(opponent_accel=3000.0)()

phase_b_ckpt = "models/stage2_scratch/phaseB/best_model.pt"
model.load_state_dict(
    torch.load(phase_b_ckpt, map_location=device, weights_only=False)
)
print(f"✅ Loaded Phase B Checkpoint from {phase_b_ckpt}")

train_ppo_vectorized(
    envs=train_envs_c,
    eval_env=eval_env_c,
    model=model,
    device=device,
    total_timesteps=10_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    eval_freq=100_000,
    eval_episodes=50,
    save_dir="models/stage2_scratch/phaseC",
    lr_initial=5e-5,
    lr_final=5e-6,
    ent_coef_initial=0.002,
    ent_coef_final=0.0002,
)

train_envs_c.close()
eval_env_c.close()

⚡ Device: cuda

🌱 PHASE A: Training From Scratch vs Slow Bot (Accel: 1600.0)
🚀 Training (80 dims) | Benchmark every 100000 steps...

📊 [EVALUATION @ Step  102400] Scored:   0.0% (0/50) | Conceded:  28.0% (14/50) | Net: -14 | Touch:  64.0% | Avg Steps: 501.7 | Mean Reward: -20.56
   ⭐ New verified best model saved: models/stage2_scratch/phaseA/best_model.pt
      [Net: -14 | Scored: 0.0% | Reward: -20.56 | Speed: 501.7 steps]


📊 [EVALUATION @ Step  200704] Scored:   0.0% (0/50) | Conceded:   8.0% (4/50) | Net:  -4 | Touch:  94.0% | Avg Steps: 580.0 | Mean Reward:  -8.34
   ⭐ New verified best model saved: models/stage2_scratch/phaseA/best_model.pt
      [Net: -4 | Scored: 0.0% | Reward: -8.34 | Speed: 580.0 steps]


📊 [EVALUATION @ Step  303104] Scored:   0.0% (0/50) | Conceded:  24.0% (12/50) | Net: -12 | Touch:  18.0% | Avg Steps: 518.1 | Mean Reward: -18.92

📊 [EVALUATION @ Step  401408] Scored:   2.0% (1/50) | Conceded:  30.0% (15/50) | Net: -14 | Touch:  68.0% | Avg Steps: 508.7 |

In [22]:

# ─────────────────────────────────────────────────────────────
# PHASE D: Full Match Intensity Bot (3200 Accel)
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("🔥 PHASE D: Full Speed Bot (Accel: 3200.0)")
print("=" * 60)

train_envs_d = gym.vector.AsyncVectorEnv(
    [make_fixed_sparring_env(opponent_accel=3200.0) for _ in range(NUM_ENVS)]
)
eval_env_d = make_fixed_sparring_env(opponent_accel=3200.0)()

phase_c_ckpt = "models/stage2_scratch/phaseC/best_model.pt"
model.load_state_dict(
    torch.load(phase_c_ckpt, map_location=device, weights_only=False)
)
print(f"✅ Loaded Phase C Checkpoint from {phase_c_ckpt}")

train_ppo_vectorized(
    envs=train_envs_d,
    eval_env=eval_env_d,
    model=model,
    device=device,
    total_timesteps=20_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    eval_freq=100_000,
    eval_episodes=50,
    save_dir="models/stage2_scratch/phaseD",
    lr_initial=5e-5,
    lr_final=5e-6,
    ent_coef_initial=0.002,
    ent_coef_final=0.0002,
)


🔥 PHASE D: Full Speed Bot (Accel: 3200.0)
✅ Loaded Phase C Checkpoint from models/stage2_scratch/phaseC/best_model.pt
🚀 Training (80 dims) | Benchmark every 100000 steps...

📊 [EVALUATION @ Step  102400] Scored:   4.0% (2/50) | Conceded:  10.0% (5/50) | Net:  -3 | Touch: 100.0% | Avg Steps: 580.6 | Mean Reward:  -9.00
   ⭐ New verified best model saved: models/stage2_scratch/phaseD/best_model.pt
      [Net: -3 | Scored: 4.0% | Reward: -9.00 | Speed: 580.6 steps]


📊 [EVALUATION @ Step  200704] Scored:  16.0% (8/50) | Conceded:  12.0% (6/50) | Net:  +2 | Touch: 100.0% | Avg Steps: 546.8 | Mean Reward:   9.99
   ⭐ New verified best model saved: models/stage2_scratch/phaseD/best_model.pt
      [Net: +2 | Scored: 16.0% | Reward: 9.99 | Speed: 546.8 steps]


📊 [EVALUATION @ Step  303104] Scored:  10.0% (5/50) | Conceded:   2.0% (1/50) | Net:  +4 | Touch: 100.0% | Avg Steps: 574.5 | Mean Reward:  11.74
   ⭐ New verified best model saved: models/stage2_scratch/phaseD/best_model.pt
      [Net

In [6]:

# ─────────────────────────────────────────────────────────────
# PHASE E: Full Match Intensity Bot (3400 Accel)
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("🔥 PHASE D: Full Speed Bot (Accel: 3400.0)")
print("=" * 60)

train_envs_e = gym.vector.AsyncVectorEnv(
    [make_fixed_sparring_env(opponent_accel=3400.0) for _ in range(NUM_ENVS)]
)
eval_env_e = make_fixed_sparring_env(opponent_accel=3400.0)()
model = ActorCritic(obs_dim=OBS_DIM).to(device)
phase_d_ckpt = "models/stage2_scratch/phaseD/best_model.pt"
model.load_state_dict(
    torch.load(phase_d_ckpt, map_location=device, weights_only=False)
)
print(f"✅ Loaded Phase D Checkpoint from {phase_d_ckpt}")

train_ppo_vectorized(
    envs=train_envs_e,
    eval_env=eval_env_e,
    model=model,
    device=device,
    total_timesteps=5_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    eval_freq=100_000,
    eval_episodes=50,
    save_dir="models/stage2_scratch/phaseE",
    lr_initial=5e-5,
    lr_final=5e-6,
    ent_coef_initial=0.002,
    ent_coef_final=0.0002,
)


🔥 PHASE D: Full Speed Bot (Accel: 3400.0)
✅ Loaded Phase D Checkpoint from models/stage2_scratch/phaseD/best_model.pt
🚀 Training (80 dims) | Benchmark every 100000 steps...

📊 [EVALUATION @ Step  102400] Scored:  84.0% (42/50) | Conceded:   0.0% (0/50) | Net: +42 | Touch: 100.0% | Avg Steps: 453.9 | Mean Reward: 114.51
   ⭐ New verified best model saved: models/stage2_scratch/phaseE/best_model.pt
      [Net: +42 | Scored: 84.0% | Reward: 114.51 | Speed: 453.9 steps]


📊 [EVALUATION @ Step  200704] Scored:  74.0% (37/50) | Conceded:   0.0% (0/50) | Net: +37 | Touch: 100.0% | Avg Steps: 463.8 | Mean Reward: 102.47

📊 [EVALUATION @ Step  303104] Scored:  30.0% (15/50) | Conceded:   6.0% (3/50) | Net: +12 | Touch: 100.0% | Avg Steps: 517.7 | Mean Reward:  38.85

📊 [EVALUATION @ Step  401408] Scored:  56.0% (28/50) | Conceded:   2.0% (1/50) | Net: +27 | Touch: 100.0% | Avg Steps: 496.4 | Mean Reward:  77.11

📊 [EVALUATION @ Step  503808] Scored:  68.0% (34/50) | Conceded:   0.0% (0/50) | N

# Testing

In [4]:
import torch
from src.rl.ppo_core import ActorCritic
from src.rl.benchmarker import RLController, run_arena, run_solo_drill, render_match
from config.match_config import PlayerSlot, PlayerStats
from src.bots.heuristic_bot import TeamHeuristicCoordinator
from src.engine.controllers import HeuristicBotController

device = torch.device("cpu") # Fast inference on CPU

# 1. Load RL models
obs_dim = 80
stage2_model = ActorCritic(obs_dim).to(device)
stage2_model.load_state_dict(torch.load("models/stage2_scratch/phaseE/best_model.pt", map_location=device))

phase_d_model = ActorCritic(obs_dim).to(device)
phase_d_model.load_state_dict(
    torch.load("models/stage2_scratch/phaseD/best_model.pt")
)

# 2. Setup Team Coordinators
red_rl_controller = RLController(stage2_model, team="red", device=device)

red_heuristic_coord = TeamHeuristicCoordinator(team="red")
red_heuristic_controller = HeuristicBotController(red_heuristic_coord)

blue_heuristic_coord = TeamHeuristicCoordinator(team="blue")
blue_heuristic_controller = HeuristicBotController(blue_heuristic_coord)

blue_rl_controller = RLController(phase_d_model, team="blue")



In [8]:
# ==========================================
# TEST 1: The Diagnostic Solo Drill
# ==========================================
print("--- TEST 1: EMPTY NET DIAGNOSTIC ---")
run_solo_drill(
    agent_roster=[PlayerSlot("red", PlayerStats("RL_Test"), red_rl_controller)],
    num_episodes=5,
    time_limit=60.0
)


--- TEST 1: EMPTY NET DIAGNOSTIC ---
🎯 Running Solo Drill: 60.0s per episode (5 Episodes)
   Episode 1: 1 goals
   Episode 2: 1 goals
   Episode 3: 0 goals
   Episode 4: 0 goals
   Episode 5: 0 goals
📊 Average Scoring Rate: 0.40 goals / 60.0s



0.4

In [6]:
# ==========================================
# TEST 2: The Arena 
# RL Agent (Red) vs Heuristic Bot (Blue)
# ==========================================
print("\n--- TEST 2: THE ARENA (1v1) ---")
red_roster = [PlayerSlot("red", PlayerStats("RL_Agent"), red_rl_controller)]
blue_roster = [PlayerSlot("blue", PlayerStats("Bot"), blue_rl_controller)]

stats = run_arena(
    red_roster=red_roster,
    blue_roster=blue_roster,
    num_matches=1,
    time_limit=60.0,  # 60 second matches
    score_limit=3
)




--- TEST 2: THE ARENA (1v1) ---
🏟️ Running Arena: 1 RED vs 1 BLUE (1 Matches)
✅ Completed in 8.93s
🏆 Series Outcome (Wins): RED 0 | BLUE 0 | DRAWS 1
⚽ Avg Goals / Match:     RED 0.00 | BLUE 0.00



In [8]:
# ==========================================
# TEST 3: Kaggle-Style Visualization
# Watch the matchup in HTML format
# ==========================================
red_roster = [PlayerSlot("red", PlayerStats("RL_Agent"), red_rl_controller)]
blue_roster = [PlayerSlot("blue", PlayerStats("Bot"), blue_heuristic_controller)]


print("\n--- TEST 3: RENDER MATCH ---")
render_match(
    red_roster=red_roster,
    blue_roster=blue_roster,
    num_matches=1,
    time_limit=90.0,
    save_path="renders/arena"
)



--- TEST 3: RENDER MATCH ---
🎬 Generating 1 replays...
Game 1 Result: RED WINS! 🎉 (3 - 1)
Replay saved to: renders/arena/2026-08-24_04-24-48_match_1.html



In [31]:
from src.rl.benchmarker import render_solo_drill


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
stage2_model.to(device)

agent_slot = PlayerSlot("red", PlayerStats("RL_Agent"), RLController(stage2_model, team="red", device=device))

# Render 3 randomized episodes (20 seconds each)
replay_path = render_solo_drill(
    agent_slot=agent_slot,
    num_episodes=3,
    time_limit=20.0,
    save_path="renders/solo_drills",
)

🎬 Generating 3 Solo Drill Replays (20.0s each)...
   Episode 1 Finished: 0 Goals Scored
   Episode 2 Finished: 0 Goals Scored
   Episode 3 Finished: 1 Goals Scored
🏆 Overall: 0.33 Avg Goals / 20.0s
Replay saved to: renders/solo_drills/2026-08-23_22-10-11_solo_drill.html

